<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [3]:
import warnings

warnings.filterwarnings("ignore")

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [5]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location='file:c:/Users/ygor1/OneDrive/Documentos/GitHub/atividade-04-deep-learning-i-YgoRosa/notebooks/mlruns/1', creation_time=1779493834785, experiment_id='1', last_update_time=1779493834785, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

Respostas

1. (32, 32, 3) → 32 pixels de altura, 32 de largura, 3 canais RGB.
2. 3072 features (32 × 32 × 3).
3. Porque a MLP recebe entrada em forma de vetor 1D, não em formato de imagem 3D.
4. Coloca os valores na mesma escala (0 a 1), melhora a estabilidade e acelera o treinamento.

**Solução**:

In [6]:
import sys
sys.path.append("..")

from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split
from src.utils import normalize_images


def load_data(seed):

    (X, y), (_, _) = cifar10.load_data()

    X = X.reshape(X.shape[0], -1)

    X = normalize_images(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_val, y_train, y_val

X_train, X_test, y_train, y_test = load_data(seed=42)

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

Respostas

1. A primeira camada possui (3072 × n) + n parâmetros, onde 3072 é o número de entradas da imagem CIFAR-10, n é o número de neurônios da primeira camada oculta e o + n corresponde aos bias. Exemplo: se tiver 128 neurônios, então (3072 × 128) + 128 = 393344 parâmetros.
2. A ReLU (f(x)=max(0,x)) serve para introduzir não-linearidade na rede, permitindo que o modelo aprenda padrões complexos, além de ajudar a reduzir o problema de gradientes pequenos e acelerar o treinamento.
3. MLPs possuem muitos parâmetros em imagens porque cada neurônio da camada oculta se conecta com todas as entradas da imagem, fazendo o número de pesos crescer muito (entradas × neurônios), o que aumenta memória, tempo de treinamento e risco de overfitting.

**Solução**:

In [7]:
from sklearn.neural_network import MLPClassifier

def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed
    )

    model.fit(X_train, y_train.ravel())

    return model

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

Respostas

1. Accuracy representa a proporção de previsões corretas feitas pelo modelo em relação ao total de previsões, mostrando a taxa geral de acerto.
2. Precision mede, entre as previsões positivas feitas pelo modelo, quantas estavam corretas, enquanto recall mede, entre os casos realmente positivos, quantos o modelo conseguiu identificar.
3. O f1-score é importante quando se quer equilibrar precision e recall, principalmente em problemas com classes desbalanceadas, onde accuracy sozinha pode ser enganosa.

**Solução**:

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1_score": f1_score(y_test, y_pred, average="weighted")
    }

    return pd.DataFrame([results])

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

Respostas 

1. O experimento com melhor desempenho foi o que apresentou maiores valores de accuracy e f1-score entre os testes realizados.
2. A configuração mais estável foi aquela que manteve métricas consistentes, sem grandes variações entre diferentes execuções.
3. O rastreamento experimental permite registrar parâmetros e métricas automaticamente, facilitando comparação, reprodutibilidade e escolha do melhor modelo.

**Solução**:

In [9]:
import time
import mlflow

def run_experiment(
    X_train,
    y_train,
    X_test,
    y_test,
    activation,
    hidden_layers,
    learning_rate,
    max_iter,
    batch_size,
    seed
):
    with mlflow.start_run():

        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed
        )

        model.fit(X_train, y_train.ravel())

        training_time = time.time() - start_time

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average="weighted")
        recall = recall_score(y_test, y_pred, average="weighted")
        f1 = f1_score(y_test, y_pred, average="weighted")

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("training_time", training_time)

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "training_time": training_time
        }

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

Respostas 

1. A ativação logistic apresentou melhor convergência, pois obteve os melhores valores de accuracy (0.395) e f1-score (0.3849) entre os experimentos realizados.
2. A ativação tanh apresentou maior estabilidade, pois teve tempo de treinamento próximo ao logistic (8.74s) e desempenho relativamente consistente, sem grande queda nas métricas.
3. Não houve diferenças muito grandes no tempo de treinamento, já que todas ficaram próximas de 9 segundos, mas houve diferenças nas métricas, com logistic apresentando melhor desempenho e relu o pior resultado neste experimento.
4. A ReLU é amplamente utilizada em Deep Learning porque é computacionalmente eficiente, simples de calcular e ajuda a reduzir o problema de vanishing gradient, o que normalmente acelera o treinamento em redes mais profundas, mesmo que neste experimento específico não tenha sido a melhor.

**Solução**:

In [10]:
activations = ["logistic", "tanh", "relu"]

results = []

# usa só uma amostra pra não ficar 3 horas
X_train_small = X_train[:5000]
y_train_small = y_train[:5000]

X_test_small = X_test[:1000]
y_test_small = y_test[:1000]

for activation in activations:
    metrics = run_experiment(
        X_train_small,
        y_train_small,
        X_test_small,
        y_test_small,
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
        max_iter=20,
        batch_size=64,
        seed=42
    )

    metrics["activation"] = activation
    results.append(metrics)

pd.DataFrame(results)

,accuracy,precision,recall,f1_score,training_time,activation
0,0.395,0.399298,0.395,0.384874,9.026882,logistic
1,0.379,0.373722,0.379,0.366844,8.740504,tanh
2,0.359,0.348198,0.359,0.330332,9.533071,relu


# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

Respostas 

1. Não, redes maiores não melhoraram sempre de forma significativa. Houve aumento gradual de accuracy, mas o ganho entre (128, 64) (0.381) e (256, 128) (0.386) foi pequeno em comparação ao aumento do custo computacional.
2. A arquitetura (128, 64) apresentou o melhor tradeoff, pois teve bom desempenho (accuracy 0.381) com custo computacional muito menor que (256, 128), que demorou quase 4 vezes mais para um ganho mínimo.
3. Não houve sinais claros de overfitting apenas olhando essas métricas, mas o pequeno ganho em accuracy com grande aumento de complexidade sugere que redes maiores podem começar a ter capacidade excessiva sem melhoria proporcional.
4. A arquitetura (256, 128) apresentou o maior custo computacional, com tempo de treinamento de 58.37 segundos, sendo a mais pesada entre todas as testadas.

**Solução**:

In [14]:
architectures = [(32,), (64,), (128, 64), (256, 128)]

results = []

for architecture in architectures:
    metrics = run_experiment(
        X_train_small,
        y_train_small,
        X_test_small,
        y_test_small,
        activation="relu",
        hidden_layers=architecture,
        learning_rate=0.001,
        max_iter=20,
        batch_size=64,
        seed=42
    )

    metrics["architecture"] = str(architecture)
    results.append(metrics)

pd.DataFrame(results)

,accuracy,precision,recall,f1_score,training_time,architecture
0,0.304,0.289773,0.304,0.272553,2.669666,"(32,)"
1,0.359,0.348198,0.359,0.330332,8.271042,"(64,)"
2,0.381,0.420005,0.381,0.365119,14.801571,"(128, 64)"
3,0.386,0.400705,0.386,0.376558,58.365980,"(256, 128)"


# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

Respostas 
1. O learning rate 0.001 apresentou melhor desempenho, pois obteve a maior accuracy (0.322) e o melhor f1-score (0.3154) entre os experimentos realizados.
2. O learning rate 0.1 apresentou maior instabilidade, pois teve o pior desempenho (accuracy 0.122 e f1-score 0.0265), indicando dificuldade de convergência.
3. Quando o learning rate é muito alto, o modelo pode dar passos muito grandes durante a otimização, “pulando” o mínimo da função de loss, causando instabilidade e pior desempenho.
4. Quando o learning rate é muito baixo, o modelo aprende de forma mais lenta, demorando mais para convergir e podendo exigir mais épocas para atingir bons resultados.



In [15]:
learning_rates = [0.1, 0.01, 0.001]

results = []

for lr in learning_rates:
    metrics = run_experiment(
        X_train[:3000],
        y_train[:3000],
        X_test[:500],
        y_test[:500],
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=10,
        batch_size=64,
        seed=42
    )

    metrics["learning_rate"] = lr
    results.append(metrics)

pd.DataFrame(results)

,accuracy,precision,recall,f1_score,training_time,learning_rate
0,0.122,0.014884,0.122,0.026531,7.456261,0.100
1,0.286,0.285692,0.286,0.254066,7.792848,0.010
2,0.322,0.333568,0.322,0.315407,8.055119,0.001


# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

Durante os experimentos foi possível observar que o comportamento da loss e das métricas depende diretamente dos hiperparâmetros escolhidos. Configurações mais adequadas apresentaram melhor convergência, enquanto escolhas inadequadas resultaram em menor accuracy e treinamento menos eficiente.

O learning rate teve grande impacto no aprendizado. Um valor muito alto (0.1) apresentou baixo desempenho (accuracy = 0.122), indicando instabilidade e dificuldade de convergência, pois os pesos provavelmente davam passos grandes demais na otimização. Já um valor muito baixo (0.001) apresentou melhor desempenho (accuracy = 0.322), mostrando aprendizado mais estável, embora com convergência mais lenta. O valor intermediário (0.01) teve desempenho intermediário.

A arquitetura da rede também influenciou os resultados. Redes menores, como (32,), tiveram baixo desempenho (accuracy = 0.304), mostrando capacidade limitada de aprendizado. Arquiteturas maiores melhoraram os resultados, chegando a accuracy = 0.386 em (256, 128). Porém, o custo computacional aumentou bastante, com tempo de treinamento subindo de aproximadamente 2.7s para 58.4s, mostrando que redes maiores nem sempre compensam pelo pequeno ganho de desempenho.

As funções de ativação apresentaram diferenças de desempenho. A ativação logistic obteve o melhor resultado (accuracy = 0.395), seguida de tanh (0.379) e relu (0.359). Além disso, logistic e tanh apresentaram comportamento mais estável, enquanto relu teve desempenho inferior nesse conjunto específico.

No comportamento geral do treinamento, foi observado que a MLP consegue aprender padrões, mas sua performance depende bastante do ajuste dos hiperparâmetros. Pequenas mudanças em ativação, arquitetura ou learning rate impactaram significativamente accuracy e estabilidade.

As MLPs possuem limitações para imagens, pois tratam a entrada como um vetor simples, sem preservar a estrutura espacial dos pixels. Isso faz com que a rede não aproveite informações locais e padrões visuais como bordas, texturas e formas, algo que arquiteturas como CNNs conseguem explorar melhor.

O backpropagation é o mecanismo responsável pelo aprendizado da rede, pois calcula o erro da saída e o propaga para trás pela rede, ajustando os pesos de cada neurônio através do gradiente. Esse processo permite que a MLP reduza a loss progressivamente e melhore sua capacidade de generalização.

Respostas

1. Qual configuração apresentou melhor resultado final?
A melhor configuração observada foi a que utilizou ativação logistic, pois apresentou a maior accuracy (0.395). Entre as arquiteturas, (256, 128) teve o melhor resultado estrutural (0.386), mas com custo computacional elevado.

2. Quais foram as principais dificuldades observadas?
As principais dificuldades foram o ajuste de hiperparâmetros, a variação de desempenho entre configurações e o aumento significativo do tempo de treinamento em redes maiores.

3. Por que MLPs possuem limitações para imagens?
Porque a MLP transforma a imagem em um vetor simples e perde a informação espacial entre pixels, dificultando a captura de padrões locais importantes.

4. Como o backpropagation contribui para o aprendizado da rede?
O backpropagation calcula o erro da rede e ajusta os pesos com base nos gradientes, permitindo que o modelo aprenda progressivamente e reduza a loss ao longo do treinamento.